# PE6201 Course Project — AI Assistant for Company Policies
**Yichen Tang · Section A**

An employee asks an HR question in plain English. The assistant answers in a sentence or two, quotes the handbook clause it relied on, shows the document date, and refuses when the handbook does not cover the question.

This notebook runs the whole project end to end: load the frozen corpus, run the Ctrl-F baseline, run the assistant on the 36-question test set, and print the two scores with counts and the cost per question.

## 1 · Setup
Run all cells. An OpenRouter API key is needed for sections 4–6; sections 2–3 run without one. In Colab, save the key as a Secret named `OPENROUTER_API_KEY` (key icon in the left sidebar) with notebook access on; otherwise the notebook asks for it.

In [ ]:
REPO_URL = "https://github.com/TyyCheN/pe6201-policy-assistant.git"

import os, sys
if not os.path.exists("src/policy_assistant"):          # running in Colab: clone the repo
    !git clone -q $REPO_URL policy-assistant
    %cd policy-assistant
!pip install -q -r requirements.txt
sys.path.insert(0, "src")

In [ ]:
import os, getpass

key = os.environ.get("OPENROUTER_API_KEY")
if not key:
    try:                                     # Colab: read the key from Secrets (key icon, left sidebar)
        from google.colab import userdata
        key = userdata.get("OPENROUTER_API_KEY")
    except Exception:
        key = None
if not key:                                  # no secret set: ask for it
    key = getpass.getpass("OpenRouter API key: ")
os.environ["OPENROUTER_API_KEY"] = key.strip()
os.environ.setdefault("OPENROUTER_MODEL", "openai/gpt-4o-mini")

## 2 · The data: one handbook, frozen before the questions were written
The corpus is the Haida Co. Employee Handbook (October 2026 revision), translated into English, split into 12 documents and 175 numbered clauses. `MANIFEST.json` holds a hash of every file; the check below fails if anything changed after the freeze commit.

In [ ]:
!python scripts/freeze_manifest.py --check
!git log --oneline --reverse | head -3

from policy_assistant import load_corpus
clauses = load_corpus()
print(len(clauses), "clauses")
print(clauses[100].id, "|", clauses[100].doc_title, "|", clauses[100].text[:160], "...")

## 3 · Ctrl-F baseline
What an employee gets by searching the handbook for the words in the question: literal word matching, no model. Same 36 questions as the assistant.

In [ ]:
from policy_assistant import evaluate as ev
from policy_assistant.keyword_baseline import KeywordBaseline

questions = ev.load_questions()
base_rows = ev.run_baseline(KeywordBaseline(clauses), questions)
b = ev.summarise(base_rows, "correct")
print("Accuracy on answerable:      ", ev.pct(b["correct"], b["answerable_n"]))
print("Refusal rate on unanswerable:", ev.pct(b["refused"], b["unanswerable_n"]))

## 4 · The assistant
Retrieve the 5 most similar clauses → **G1** refuse if the best one is below a threshold set on the dev set → **G2** the model must answer only from those clauses and say when they do not cover the question → **G3** the quoted text must appear word for word in the cited clause → **G4** the answer is shown with clause ID, chapter and document date.

In [ ]:
from policy_assistant import Retriever, PolicyAssistant, OpenRouterLLM
from policy_assistant.assistant import format_answer

retriever = Retriever(clauses)
tau, dev_scores = ev.calibrate_tau(retriever, ev.load_questions("dev.jsonl"))
print("backend:", retriever.backend, "| tau:", tau, "|", dev_scores)
llm = OpenRouterLLM()
assistant = PolicyAssistant(retriever, llm, tau=tau, k=5)

for q in ["How many days of marriage leave do I get?",
          "My wife is about to give birth. How many days off can I take as the father?",
          "How much is the monthly meal allowance?",
          "What is Zhang Wei's salary this month?"]:
    print("Q:", q)
    print(format_answer(assistant.ask(q)))
    print("-" * 80)

## 5 · Evaluation: two scores, with counts
24 answerable questions → accuracy. 12 unanswerable questions → refusal rate. The L2 judge compares each answer with the hand-written gold answer.

In [ ]:
from pathlib import Path
from IPython.display import Markdown

sys_rows = ev.run_assistant(assistant, questions, judge_llm=llm)
Path("results").mkdir(exist_ok=True)
ev.write_csv(sys_rows, "results/assistant.csv")
ev.write_csv(base_rows, "results/baseline.csv")
meta = {"model": llm.model, "backend": retriever.backend, "tau": tau, "k": 5, "clauses": len(clauses)}
summary = ev.report(sys_rows, base_rows, meta)
Path("results/summary.md").write_text(summary, encoding="utf-8")
Markdown(summary)

## 6 · Every question, side by side
For reading the failures, not just counting them.

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 90)
df = pd.DataFrame(sys_rows)[["id", "type", "question", "refused", "gate", "answer", "cited", "gold", "l1_correct", "l2_correct"]]
df["ctrl_f_top1"] = [r["top1"] or "(refused)" for r in base_rows]
df

## 7 · Try your own question

In [ ]:
# Change the question and run this cell again.
question = "I was 3 minutes late twice this month. Do I lose my full-attendance bonus?"
print(format_answer(assistant.ask(question)))